<a href="https://colab.research.google.com/github/nicolobagnoli/Structure-Aware-Transformers/blob/GNN/GNN_Cora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#https://www.youtube.com/watch?v=8owQBFAHw7E&list=WL&index=35&t=1602s

!pip install numpy
!pip install tensorflow
!pip install spektral

import numpy as np
import tensorflow as tf
import spektral

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 2.6 MB/s eta 0:00:00


In [3]:
print(spektral.__version__)
from spektral.datasets import Cora

1.3.1


In [4]:
#load the dataset
dataset = Cora()

#extract data
adj = dataset[0].a         # Adjacency matrix
features = dataset[0].x    # Node features
labels = dataset[0].y      # Labels

#train, validation, and test masks
train_mask = dataset.mask_tr
val_mask = dataset.mask_va
test_mask = dataset.mask_te

print("Adjacency matrix shape:", adj.shape)
print("Feature matrix shape:", features.shape)
print("Labels shape:", labels.shape)

Adjacency matrix shape: (2708, 2708)
Feature matrix shape: (2708, 1433)
Labels shape: (2708, 7)


In [5]:
from scipy.sparse import issparse

if issparse(features):
    features = features.todense()   #convertes matrices with many zero values to data format that is more dense

if issparse(adj):
    adj = adj.todense()             #converts matrices with many zero values to data format that is more dense
adj = adj + np.eye(adj.shape[0]) #to ensure each node is connected to itself!

features = features.astype(np.float32)  #many deep learing models excpect data in this format
adj = adj.astype(np.float32)

#just bc im curious
print (np.sum(train_mask))
print (np.sum(val_mask))
print (np.sum(test_mask))

140
500
1000


In [6]:
def masked_softmax_cross_entropy(logits, labels, mask):
  #computes softmax cross entropy loss for each sample in logits compared to labels
  loss = tf.nn.softmax_cross_entropy_with_logits(logits=logits, labels=labels)
  #simply converting mask to float
  mask = tf.cast(mask, dtype=tf.float32)
  #normalizes mask
  mask /= tf.reduce_mean(mask)
  #computes average cross entropy loss over all masked elements
  return tf.reduce_mean(loss)

def masked_accuracy(logits, labels, mask):
  #tf.equal compares predicted and true class of logits and labels returning true/false
  correct_prediction = tf.equal(tf.argmax(logits, 1), tf.argmax(labels, 1))
  #converts true to 1 and false to 0 for accuracy calculations
  accuracy_all = tf.cast(correct_prediction, tf.float32)
  #converts mask to float
  mask = tf.cast(mask, dtype=tf.float32)
  #normalizes
  mask /= tf.reduce_mean(mask)
  #applies the mask so only valid datapoints contribute to the accuracy
  accuracy_all *= mask
  #computes the avg. accuracy of only masked elements
  return tf.reduce_mean(accuracy_all)

In [7]:
#we transform each of the nodes with a point wise transformation, and once we have these feature that we want to aggregate we perform a matrix multiplication between the adjecentcy matrix and these features

def gnn_fn(fts, adj, transform, activation):
  seq_fts = transform(fts)
  ret_fts = tf.matmul(adj, seq_fts)
  return activation(ret_fts)

In [8]:
import tensorflow as tf

#define feature and adjacency matrix
fts = features  # Assuming 'features' is defined
adj = adj  # Assuming 'adj' is defined

#define GNN layers globally so they are accessible inside cora_gnn
units = 512
lyr_1 = tf.keras.layers.Dense(units)
lyr_2 = tf.keras.layers.Dense(7)  # 7 classes for Cora dataset

#define GNN function
def cora_gnn(fts, adj, gnn_fn):
    hidden = gnn_fn(fts, adj, lyr_1, tf.nn.relu)
    logits = gnn_fn(hidden, adj, lyr_2, tf.identity)
    return logits

#training Parameters
lr = 1e-2
epochs = 40
batch_size = 140
display_step = 1

optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

best_accuracy = 0.0
for ep in range(epochs + 1):
    with tf.GradientTape() as t:
        logits = cora_gnn(fts, adj, gnn_fn)  #pass gnn_fn with identity matrix
        loss = masked_softmax_cross_entropy(logits, labels, train_mask)

        variables = t.watched_variables()
        grads = t.gradient(loss, variables)
        optimizer.apply_gradients(zip(grads, variables))

        #compute validation and test accuracy
        logits = cora_gnn(fts, adj, gnn_fn)
        val_accuracy = masked_accuracy(logits, labels, val_mask)
        test_accuracy = masked_accuracy(logits, labels, test_mask)

        #track best validation accuracy
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            print('Epoch', ep, '| Training loss:', loss.numpy(), '| Val accuracy:', val_accuracy.numpy(), '|Test accuracy:', test_accuracy.numpy())


Epoch 0 | Training loss: 4.9808497 | Val accuracy: 0.218 |Test accuracy: 0.203
Epoch 1 | Training loss: 38.914165 | Val accuracy: 0.29999998 |Test accuracy: 0.308
Epoch 2 | Training loss: 32.459995 | Val accuracy: 0.54999995 |Test accuracy: 0.538
Epoch 3 | Training loss: 22.940733 | Val accuracy: 0.644 |Test accuracy: 0.63799995
Epoch 4 | Training loss: 12.588929 | Val accuracy: 0.64599997 |Test accuracy: 0.663
Epoch 5 | Training loss: 9.487152 | Val accuracy: 0.72599995 |Test accuracy: 0.741
Epoch 6 | Training loss: 6.090152 | Val accuracy: 0.782 |Test accuracy: 0.806
Epoch 7 | Training loss: 4.0995765 | Val accuracy: 0.79199994 |Test accuracy: 0.816
Epoch 12 | Training loss: 2.7241266 | Val accuracy: 0.80399996 |Test accuracy: 0.82600003
Epoch 13 | Training loss: 2.4325342 | Val accuracy: 0.812 |Test accuracy: 0.8320001
Epoch 14 | Training loss: 2.2179947 | Val accuracy: 0.83199996 |Test accuracy: 0.8390001
Epoch 15 | Training loss: 1.9075776 | Val accuracy: 0.862 |Test accuracy: 0.8